<a href="https://colab.research.google.com/github/rahmanullahkhan123/Generative_AI/blob/main/Autonomous_Multi_Agent_AI_Research_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Mon Jul 27 11:28:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install langchain
!pip install langgraph
!pip install langchain-community
!pip install langchain-huggingface
!pip install transformers
!pip install sentence-transformers
!pip install chromadb
!pip install pypdf
!pip install duckduckgo-search
!pip install accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [3]:
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

from langchain_core.prompts import PromptTemplate

from langgraph.graph import StateGraph, END

from typing import TypedDict

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM


model_name = "mistralai/Mistral-7B-Instruct-v0.2"


tokenizer = AutoTokenizer.from_pretrained(
    model_name
)


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [5]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512
)


llm = HuggingFacePipeline(
    pipeline=pipe
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [6]:
class AgentState(TypedDict):

    question:str
    plan:str
    research:str
    summary:str
    report:str

In [7]:
def planner_agent(state):

    prompt = f"""

    Break this research topic into tasks:

    {state['question']}

    """

    result = llm.invoke(prompt)


    return {
        "plan": result
    }

In [8]:
def research_agent(state):

    prompt = f"""

    Research these tasks:

    {state['plan']}

    """

    result = llm.invoke(prompt)


    return {
        "research":result
    }

In [9]:
def summary_agent(state):

    prompt=f"""

    Summarize:

    {state['research']}

    """

    result=llm.invoke(prompt)


    return {
        "summary":result
    }

In [10]:
def report_agent(state):

    prompt=f"""

    Create professional report:

    {state['summary']}

    """

    result=llm.invoke(prompt)


    return {
        "report":result
    }

In [11]:
workflow = StateGraph(AgentState)


workflow.add_node(
    "planner",
    planner_agent
)


workflow.add_node(
    "research",
    research_agent
)


workflow.add_node(
    "summary",
    summary_agent
)


workflow.add_node(
    "report",
    report_agent
)



workflow.set_entry_point(
    "planner"
)


workflow.add_edge(
    "planner",
    "research"
)


workflow.add_edge(
    "research",
    "summary"
)


workflow.add_edge(
    "summary",
    "report"
)


workflow.add_edge(
    "report",
    END
)



app = workflow.compile()

In [12]:
result = app.invoke(
{
"question":
"Applications of Generative AI in Healthcare"
}
)


print(result["report"])

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_



    Create professional report:

    

    Summarize:

    

    Research these tasks:

    

    Break this research topic into tasks:

    Applications of Generative AI in Healthcare

    1. Research existing applications of generative AI in healthcare:
        a. Identify key use cases, such as medical image generation, drug discovery, and patient record generation.
        b. Understand the benefits and limitations of each application.
    2. Study the latest research and developments in generative AI for healthcare:
        a. Follow relevant research papers, conferences, and journals.
        b. Analyze the latest trends and advancements in the field.
    3. Evaluate the ethical and regulatory considerations of using generative AI in healthcare:
        a. Understand the legal and ethical frameworks for AI in healthcare.
        b. Analyze the potential risks and benefits of using generative AI in sensitive areas, such as patient privacy and data security.
    4. Assess the cur